# RAG Infrastructure & Pipeline
## The Architects — Member 1: Vector Database Lead + Member 2: Integration Engineer

**Scope:**
- Set up the vector store (Pinecone / Azure Cognitive Search / Milvus)
- Indexing strategy, metadata filtering, scalability
- Configure the Hugging Face pipeline
- Connect retriever (vector store) → generator (LLM)
- Prompt template logic & API orchestration

**Data source:** Preprocessed & EDA-cleaned FAQ CSV

---
## 0. Install Dependencies

In [ ]:
# Run once — restart kernel after installation
!pip install pymilvus pandas sentence-transformers transformers accelerate torch langchain langchain-community python-dotenv tqdm

In [ ]:
# for using the local gpu
!pip uninstall torch
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

---
## 1. Configuration & Credentials

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

# ─── Member 1: Vector Store & Retrieval Config ──────────────────────────────
VECTOR_STORE_BACKEND = "milvus"
MILVUS_HOST = "localhost"
MILVUS_PORT = "19530"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384  # Must match the model's output dimension
TOP_K = 3  # Number of chunks to retrieve per query

# ─── Member 2: Generator (LLM) Config ───────────────────────────────────────
GENERATOR_MODEL = "google/flan-t5-large"
MAX_NEW_TOKENS = 256

print("Configuration loaded.")
print(f"  Vector store : Local {VECTOR_STORE_BACKEND.capitalize()} at {MILVUS_HOST}:{MILVUS_PORT}")
print(f"  Embedding    : {EMBEDDING_MODEL} (dim={EMBEDDING_DIM})")
print(f"  Generator    : {GENERATOR_MODEL}")

Configuration loaded.
  Vector store : Local Milvus at localhost:19530
  Embedding    : sentence-transformers/all-MiniLM-L6-v2 (dim=384)
  Generator    : google/flan-t5-large


---
## 2. Load & Prepare Cleaned Data

> Drop constant/null columns that carry no signal (`requires_review`, `review_reason`).

In [2]:
import pandas as pd

# ── Load your cleaned CSV ────────────────────────────────────────────────────
df = pd.read_csv("../data/unified_faq_dataset.csv")

# Drop zero-signal columns
df.drop(columns=["requires_review", "review_reason"], inplace=True)


# Parse metadata_str into individual columns (category, difficulty, source)
def parse_metadata(meta_str):
    parsed_metadata = {}
    if isinstance(meta_str, str):
        for part in meta_str.split(" | "):
            if ":" in part:
                k, v = part.split(":", 1)
                parsed_metadata[k.strip()] = v.strip()
    return parsed_metadata

In [3]:
# 1. Parse the metadata into its own dataframe
meta_df = df["metadata_tags"].apply(parse_metadata).apply(pd.Series)

# 2. ONLY extract 'source' from the metadata
df['source'] = meta_df['source']

# 3. Rename difficulty_level to difficulty
df["difficulty"] = df["difficulty_level"]

# 4. Final Cleanup
df.reset_index(drop=True, inplace=True)
df["doc_id"] = df.index.astype(str)

print(f"Total Records: {len(df)}")
print(f"Categories missing: {df['category'].isnull().sum()}")
print(f"Final Columns: {df.columns.tolist()}")

# Preview the data to verify
print("\nVerification (First 3 rows):")
print(df[['doc_id', 'category', 'source', 'difficulty']].head(3))

Total Records: 53842
Categories missing: 0
Final Columns: ['category', 'difficulty_level', 'question', 'answer', 'metadata_tags', 'source', 'difficulty', 'doc_id']

Verification (First 3 rows):
  doc_id            category source    difficulty
0      0   Account & Profile    faq  Intermediate
1      1  Billing & Payments    faq         Basic
2      2   Account & Profile    faq  Intermediate


---
## 3. Construct Embedding Text

Combine `question + answer` (optionally with `category`) into a single string per row.  
This gives the embedding model full semantic context rather than just the bare answer.

In [4]:
def build_embedding_text(row, include_category=True):
    """
    Constructs the text to embed for a single FAQ row.
    
    Format:
        [Category] Q: <question> A: <answer>
    """
    prefix = f"[{row['category']}] " if include_category else ""
    return f"{prefix}Q: {row['question']} A: {row['answer']}"


df["embedding_text"] = df.apply(build_embedding_text, axis=1)

# Preview
for text in df["embedding_text"].head(3):
    print(text)
    print()

[Account & Profile] Q: How can I create an account? A: To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.

[Billing & Payments] Q: What payment methods do you accept? A: We accept major credit cards, debit cards, and PayPal as payment methods for online orders.

[Account & Profile] Q: How can I track my order? A: You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment.



---
## 4. Generate Embeddings (Member 1)

Using `sentence-transformers` locally.  
Swap `EMBEDDING_MODEL` to an OpenAI model and use the OpenAI SDK if you prefer a hosted option.

In [5]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

print(f"Loading embedding model: {EMBEDDING_MODEL} ...")
embedder = SentenceTransformer(
    EMBEDDING_MODEL,
    device="cuda",
    token=HF_TOKEN
)

texts = df["embedding_text"].tolist()

print(f"Embedding {len(texts)} documents ...")
embeddings = embedder.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True  # Cosine similarity works best with L2-normalized vectors
)

print(f"Embedding matrix shape: {embeddings.shape}")
assert embeddings.shape[1] == EMBEDDING_DIM, (
    f"Dimension mismatch: model outputs {embeddings.shape[1]}, expected {EMBEDDING_DIM}. "
    "Update EMBEDDING_DIM in the config cell."
)

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding 53842 documents ...


Batches:   0%|          | 0/1683 [00:00<?, ?it/s]

Embedding matrix shape: (53842, 384)


---
## 5. Build Metadata Payloads (Member 1)

The metadata stored alongside each vector is what enables:
- **Filtered retrieval** (e.g. only search in `category == 'Billing & Payments'`)
- **Displaying the raw Q&A** to the user after retrieval

In [6]:
metadata_list = df[["doc_id", "category", "difficulty", "source", "question", "answer"]].to_dict('records')

print(f"Built {len(metadata_list)} metadata payloads.")
print("\nSample payload:")
import json

print(json.dumps(metadata_list[0], indent=2))

Built 53842 metadata payloads.

Sample payload:
{
  "doc_id": "0",
  "category": "Account & Profile",
  "difficulty": "Intermediate",
  "source": "faq",
  "question": "How can I create an account?",
  "answer": "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process."
}


---
## 6. Vector Store Setup & Indexing (Member 1)

Run **one** of the three subsections below depending on your chosen backend.  
Set `VECTOR_STORE_BACKEND` in the config cell above.

In [7]:
from pymilvus import connections, Collection, CollectionSchema, FieldSchema, DataType, utility

# Connect to local Docker container
connections.connect("default", host=MILVUS_HOST, port=MILVUS_PORT)

COLLECTION_NAME = "faq_rag"

# Reset collection if it already exists (useful for iterative testing)
if utility.has_collection(COLLECTION_NAME):
    Collection(COLLECTION_NAME).drop()
    print(f"Dropped existing collection '{COLLECTION_NAME}'.")

# Define schema
schema = CollectionSchema([
    FieldSchema(name="doc_id", dtype=DataType.VARCHAR, max_length=100, is_primary=True),
    FieldSchema(name="category", dtype=DataType.VARCHAR, max_length=256),
    FieldSchema(name="difficulty", dtype=DataType.VARCHAR, max_length=100),
    FieldSchema(name="source", dtype=DataType.VARCHAR, max_length=100),
    FieldSchema(name="question", dtype=DataType.VARCHAR, max_length=1000),
    FieldSchema(name="answer", dtype=DataType.VARCHAR, max_length=10000),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=EMBEDDING_DIM),
], description="FAQ RAG collection")

collection = Collection(name=COLLECTION_NAME, schema=schema)

# Build HNSW index on the vector field
# Using "IP" (Inner Product) as it is the standard for normalized cosine similarity in Milvus
collection.create_index(
    field_name="embedding",
    index_params={
        "index_type": "HNSW",
        "metric_type": "IP",  # Changed from COSINE to IP
        "params": {"M": 16, "efConstruction": 200}
    }
)

# --- Batch Insertion Logic ---
BATCH_SIZE = 10000
total_rows = len(df)

print(f"Inserting {total_rows} rows in batches of {BATCH_SIZE}...")

for i in range(0, total_rows, BATCH_SIZE):
    # Calculate the end index for the current batch
    end_idx = min(i + BATCH_SIZE, total_rows)
    print(f"  -> Inserting rows {i} to {end_idx}...")

    # Slice the dataframe and the numpy array
    batch_df = df.iloc[i:end_idx]
    batch_embeddings = embeddings[i:end_idx]

    # Build the payload for just this batch
    insert_data = [
        batch_df["doc_id"].tolist(),
        batch_df["category"].tolist(),
        batch_df["difficulty"].tolist(),
        batch_df["source"].tolist(),
        batch_df["question"].tolist(),
        batch_df["answer"].tolist(),
        batch_embeddings
    ]

    # Insert the batch
    collection.insert(insert_data)

# --- Finalize ---
print("Flushing data to disk and loading into RAM...")
collection.flush()  # Ensure data is written to disk
collection.load()  # Load into memory for searching

print(f"Milvus collection '{COLLECTION_NAME}' created and loaded with {collection.num_entities} entities.")

Dropped existing collection 'faq_rag'.
Inserting 53842 rows in batches of 10000...
  -> Inserting rows 0 to 10000...
  -> Inserting rows 10000 to 20000...
  -> Inserting rows 20000 to 30000...
  -> Inserting rows 30000 to 40000...
  -> Inserting rows 40000 to 50000...
  -> Inserting rows 50000 to 53842...
Flushing data to disk and loading into RAM...
Milvus collection 'faq_rag' created and loaded with 53842 entities.


---
## 7. Retriever — Vector Search with Metadata Filtering (Member 1)

The retriever embeds a user query and fetches the top-k most relevant FAQ chunks.  
Optional metadata filters let you scope the search (e.g. by category or difficulty).

In [8]:
def retrieve(
        query: str,
        top_k: int = TOP_K,  # Ensure TOP_K is defined in your script (e.g., 3 or 5)
        filter_category: str = None,
        filter_difficulty: str = None
) -> list[dict]:
    """
    Embed `query` and return the top-k FAQ chunks from the local Milvus store.
    """
    # 1. Embed the user's question (Forced to GPU for real-time speed)
    q_vec = embedder.encode(
        query,
        normalize_embeddings=True,
        device="cuda"
    )

    # 2. Build metadata filters if requested
    expr_parts = []
    if filter_category:  expr_parts.append(f'category == "{filter_category}"')
    if filter_difficulty: expr_parts.append(f'difficulty == "{filter_difficulty}"')
    expr = " and ".join(expr_parts) if expr_parts else None

    # 3. Search the vector database
    results = collection.search(
        data=[q_vec],  # Pymilvus can handle the 1D numpy array wrapped in a list
        anns_field="embedding",
        param={"metric_type": "IP", "params": {"ef": 64}},  # MATCHES THE INDEX
        limit=top_k,
        expr=expr,
        output_fields=["doc_id", "category", "difficulty", "source", "question", "answer"]
    )

    # 4. Format results
    hits = []
    for hit in results[0]:
        entity = hit.entity.to_dict()
        entity["score"] = hit.score
        hits.append(entity)

    return hits


# ── Sanity check ──────────────────────────────────────────────────────────────
test_results = retrieve("How do I return a product?", top_k=3)
print(f"Top {len(test_results)} results for 'How do I return a product?':\n")
for r in test_results:
    print(f"  [{r.get('score', 0):.4f}] [{r['category']}] {r['question']}")

Top 3 results for 'How do I return a product?':

  [0.6892] [Returns & Refunds] Can I return a product if I changed my mind?
  [0.6843] [Returns & Refunds] Can I return a product if I no longer have the original packaging?
  [0.6434] [Returns & Refunds] Can I return a product if I no longer have the original receipt?


---
## 8. Metadata Filter Demo (Member 1)

Demonstrate scoped retrieval — same query, different filters.

In [9]:
query = "Can I cancel or change my order?"

# Unfiltered
unfiltered = retrieve(query, top_k=3)

# Category-filtered
cat_filtered = retrieve(query, top_k=3, filter_category="Orders & Shipping")

# Difficulty-filtered
diff_filtered = retrieve(query, top_k=3, filter_difficulty="Advanced")


def show_results(label, results):
    print(f"\n{'=' * 60}")
    print(f" {label}")
    print(f"{'=' * 60}")
    for r in results:
        print(f"  Score      : {r.get('score', 0):.3f}")
        print(f"  Category   : {r['category']} | Difficulty: {r['difficulty']}")
        print(f"  Question   : {r['question']}")
        print(f"  Answer     : {r['answer'][:80]}...")
        print()


show_results("No filter", unfiltered)
show_results("Filter: Orders & Shipping", cat_filtered)
show_results("Filter: Advanced only", diff_filtered)


 No filter
  Score      : 0.796
  Category   : Orders & Shipping | Difficulty: Advanced
  Question   : Can I change or cancel an item in my order?
  Answer     : If you need to change or cancel an item in your order, please contact our custom...

  Score      : 0.748
  Category   : Orders & Shipping | Difficulty: Advanced
  Question   : Can I cancel my order?
  Answer     : You can cancel your order if it has not been shipped yet. Please contact our cus...

  Score      : 0.674
  Category   : Orders & Shipping | Difficulty: Advanced
  Question   : Can I change my order after it has been placed?
  Answer     : If you need to change your order, please contact our customer support team as so...


 Filter: Orders & Shipping
  Score      : 0.796
  Category   : Orders & Shipping | Difficulty: Advanced
  Question   : Can I change or cancel an item in my order?
  Answer     : If you need to change or cancel an item in your order, please contact our custom...

  Score      : 0.748
  Category  

---
## 9. Configure the Hugging Face Generator (Member 2)

Load the LLM pipeline using `transformers`.  
Default: `google/flan-t5-base` (CPU-friendly).  
Swap to `mistralai/Mistral-7B-Instruct-v0.2` or similar for stronger generation.

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

DEVICE = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if DEVICE == 0 else 'CPU'}")

print(f"Loading generator: {GENERATOR_MODEL} ...")

# 1. Load the Tokenizer (The "Ears" that convert words to numbers)
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL)

# 2. Load the Model (The "Brain")
model = AutoModelForSeq2SeqLM.from_pretrained(
    GENERATOR_MODEL,
    dtype=torch.float16,  # Keeps your v-ram usage incredibly low
).to(DEVICE)

print("T5 Generator ready.")

Using device: GPU
Loading generator: google/flan-t5-large ...


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


T5 Generator ready.


---
## 10. Prompt Template (Member 2)

The prompt template is the glue between retriever output and generator input.  
It instructs the LLM to answer **only** from the retrieved context — reducing hallucinations.

In [17]:
# PROMPT_TEMPLATE = """\
# You are a helpful customer support assistant.
# Answer the user's question using ONLY the context provided below.
# If the answer is not in the context, say "I don't have that information."
# Be concise and professional.
#
# Context:
# {context}
#
# Question: {question}
#
# Answer:"""

# def build_context(retrieved_chunks: list[dict]) -> str:
#     """
#     Format retrieved FAQ chunks into a numbered context block.
#     """
#     lines = []
#     for i, chunk in enumerate(retrieved_chunks, 1):
#         lines.append(f"[{i}] Q: {chunk['question']}")
#         lines.append(f"    A: {chunk['answer']}")
#     return "\n".join(lines)

PROMPT_TEMPLATE = """\
Answer the question based on the context below.
If the answer cannot be found in the context, reply exactly with "I don't have that information."

Context:
{context}

Question: {question}
Answer:"""


def build_context(retrieved_chunks: list[dict]) -> str:
    lines = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        lines.append(chunk['answer'])
    return "\n\n".join(lines)


def build_prompt(question: str, retrieved_chunks: list[dict]) -> str:
    context = build_context(retrieved_chunks)
    return PROMPT_TEMPLATE.format(context=context, question=question)


# Preview prompt for a sample query
sample_query = "What payment methods do you accept?"
sample_chunks = retrieve(sample_query, top_k=2)
sample_prompt = build_prompt(sample_query, sample_chunks)
print(sample_prompt)

Answer the question based on the context below.
If the answer cannot be found in the context, reply exactly with "I don't have that information."

Context:
We accept major credit cards, debit cards, and PayPal as payment methods for online orders.

Yes, we take the security of your personal and payment details seriously. We use industry-standard encryption and follow strict security protocols to ensure your information is protected.

Question: What payment methods do you accept?
Answer:


---
## 11. End-to-End RAG Pipeline (Member 2)

Combines retriever + prompt builder + generator into a single callable.

In [18]:
def rag_answer(
        question: str,
        top_k: int = TOP_K,
        filter_category: str = None,
        filter_difficulty: str = None,
        verbose: bool = False
) -> dict:
    """
    Full RAG pipeline:
      1. Retrieve relevant FAQ chunks from the vector store.
      2. Build a grounded prompt.
      3. Generate an answer with the LLM.

    Returns
    -------
    dict with keys: question, answer, retrieved_chunks, prompt
    """
    # Step 1: Retrieve
    chunks = retrieve(
        question,
        top_k=top_k,
        filter_category=filter_category,
        filter_difficulty=filter_difficulty
    )

    # Step 2: Build prompt
    prompt = build_prompt(question, chunks)

    # Step 3: Generate
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(DEVICE)
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=2,  # Beam search for better quality
            early_stopping=True
        )

    # Decode math back to English
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    result = {
        "question": question,
        "answer": answer,
        "retrieved_chunks": chunks,
        "prompt": prompt,
    }

    if verbose:
        print(f"Question : {question}")
        print(f"Answer   : {answer}")
        print(f"\nRetrieved {len(chunks)} chunks:")
        for c in chunks:
            print(f"  [{c.get('score', 0):.3f}] {c['question']}")

    return result

In [14]:
# ── Run a test query ──────────────────────────────────────────────────────────
# rag_answer(
#     "How do I return a damaged item?",
#     verbose=True
# )
rag_answer("What payment methods do you accept?", verbose=True)

Question : What payment methods do you accept?
Answer   : major credit cards, debit cards, and PayPal.

Retrieved 3 chunks:
  [0.711] What payment methods do you accept?
  [0.392] Are my personal and payment details secure?
  [0.369] The bank is not accepting these checks. Have any of you run into this?


{'question': 'What payment methods do you accept?',
 'answer': 'major credit cards, debit cards, and PayPal.',
 'retrieved_chunks': [{'doc_id': '1',
   'distance': 0.7112768888473511,
   'entity': {'category': 'Billing & Payments',
    'difficulty': 'Basic',
    'source': 'faq',
    'question': 'What payment methods do you accept?',
    'answer': 'We accept major credit cards, debit cards, and PayPal as payment methods for online orders.',
    'doc_id': '1'},
   'score': 0.7112768888473511},
  {'doc_id': '13',
   'distance': 0.39178699254989624,
   'entity': {'category': 'Billing & Payments',
    'difficulty': 'Intermediate',
    'source': 'faq',
    'question': 'Are my personal and payment details secure?',
    'answer': 'Yes, we take the security of your personal and payment details seriously. We use industry-standard encryption and follow strict security protocols to ensure your information is protected.',
    'doc_id': '13'},
   'score': 0.39178699254989624},
  {'doc_id': '42112',


---
## 12. Batch Inference on a Test Set (Member 2)

Run the pipeline over multiple questions and collect results for handoff to the Evaluators team.

In [23]:
test_questions = [
    "How can I create an account?",
    "What payment methods do you accept?",
    "How can I track my order?",
    "What is your return policy?",
    "Can I cancel my order?",
    "How long does shipping take?",
    "Do you offer international shipping?",
    "What should I do if my package is lost?",
    "Can I change my shipping address?",
    "How can I contact customer support?",
]

batch_results = []
for q in tqdm(test_questions, desc="Running RAG inference"):
    r = rag_answer(q, top_k=TOP_K)
    batch_results.append({
        "question": r["question"],
        "generated_answer": r["answer"],
        "top1_retrieved_q": r["retrieved_chunks"][0]["question"] if r["retrieved_chunks"] else "",
        "top1_score": r["retrieved_chunks"][0].get("score", 0) if r["retrieved_chunks"] else 0,
        "top1_category": r["retrieved_chunks"][0]["category"] if r["retrieved_chunks"] else "",
    })

pd.DataFrame(batch_results)

Running RAG inference: 100%|██████████| 10/10 [00:14<00:00,  1.47s/it]


,question,generated_answer,top1_retrieved_q,top1_score,top1_category
0,How can I create an account?,Click on the 'Sign Up' button on the top right...,How can I create an account?,0.838717,Account & Profile
1,What payment methods do you accept?,"major credit cards, debit cards, and PayPal",What payment methods do you accept?,0.711277,Billing & Payments
2,How can I track my order?,by logging into your account and navigating to...,How can I track my order?,0.816749,Account & Profile
3,What is your return policy?,I don't have that information.,What is your return policy?,0.726549,Home & Kitchen
4,Can I cancel my order?,If it has not been shipped yet,Can I cancel my order?,0.816421,Orders & Shipping
5,How long does shipping take?,3-5 business days,How long does shipping take?,0.754508,Orders & Shipping
6,Do you offer international shipping?,Yes,"Hi, do you offer international shipping?",0.783682,Grocery & Gourmet Food
7,What should I do if my package is lost?,contact our customer support team immediately,What should I do if my package is lost or dama...,0.749059,Customer Support
8,Can I change my shipping address?,I don't have that information,Can I change my shipping address after placing...,0.820902,Orders & Shipping
9,How can I contact customer support?,by phone at [phone number] or by email at [ema...,How can I contact customer support?,0.606072,Customer Support


---
## 13. Scalability Checklist (Member 1 — Reference)

| Concern                     | What we've done                                                                                   | Next step if scaling                                                                                               |
|-----------------------------|---------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------------------|
| **Index type**              | **HNSW** using the **Inner Product (IP)** metric                                                  | Tune `M` and `efConstruction` for recall vs. speed trade-off                                                       |
| **Batch upsert**            | Batches of **10,000** per request                                                                 | Parallelize the insertion loop with `concurrent.futures`                                                           |
| **Metadata filtering**      | `category`, `difficulty`, `source` — accurately mapped and filterable                             | Add more fields as needed; index only filterable fields                                                            |
| **Embedding normalisation** | **L2-normalized** vectors → allowed us to use lightning-fast **IP** to simulate Cosine Similarity | Keeps retrieval fast and mathematically accurate on all backends                                                   |
| **Model swap**              | `EMBEDDING_MODEL` and `GENERATOR_MODEL` are single config vars                                    | Drop in any HuggingFace-compatible model                                                                           |
| **Infrastructure**          | **Local Milvus Docker container** (`localhost:19530`)                                             | Migrate to a managed cloud database (e.g., Zilliz Cloud for Milvus, Pinecone, or Azure) to auto-scale with traffic |